# 02. Извлечение текста из PDF

Результат: `artifacts/df_extracted.pkl`

In [ ]:
import os, warnings, pickle, time, re, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import fitz
from PIL import Image
import pytesseract

warnings.filterwarnings("ignore")

DATA_DIR = Path("data_PI")
OUT_DIR  = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

TESSERACT_CMD_CANDIDATES = [
    shutil.which("tesseract"),
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
    os.path.expanduser(r"~\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"),
]
TESSERACT_CMD = next((p for p in TESSERACT_CMD_CANDIDATES if p and Path(p).exists()), None)

if TESSERACT_CMD is None:
    raise RuntimeError("Tesseract не найден. Установите его (см. markdown выше) или задайте путь вручную")

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
print(f"Tesseract: {TESSERACT_CMD}")
print(f"Версия:    {pytesseract.get_tesseract_version()}")

available_langs = set(pytesseract.get_languages(config=""))
required_langs  = {"kaz", "rus", "eng"}
missing = required_langs - available_langs
if missing:
    raise RuntimeError(
        f"Не хватает языковых пакетов Tesseract: {missing}. "
        f"Доступно: {sorted(available_langs)}. "
        "Скачайте *.traineddata с https://github.com/tesseract-ocr/tessdata_best "
        "и положите в каталог tessdata."
    )
print(f"Языки OK: {sorted(required_langs)}")

OCR_LANG   = "kaz+rus+eng"
OCR_DPI    = 300
MIN_CHARS  = 30

Tesseract: C:\Program Files\Tesseract-OCR\tesseract.exe
Версия:    5.5.0.20241111
Языки OK: ['eng', 'kaz', 'rus']


In [3]:
pdf_paths = sorted(DATA_DIR.rglob("*.pdf"))

df = pd.DataFrame({
    "path":     [str(p) for p in pdf_paths],
    "filename": [p.name for p in pdf_paths],
    "year":     [p.parent.name for p in pdf_paths],
})

print(f"{len(df)} PDFs")

203 PDFs


In [ ]:
def extract_pages_pymupdf(path: str) -> list[dict]:
    out = []
    try:
        doc = fitz.open(path)
    except Exception:
        return out

    for page in doc:
        try:
            text = page.get_text("text") or ""
        except Exception:
            text = ""

        page_area = max(page.rect.width * page.rect.height, 1.0)
        max_img_ratio = 0.0
        try:
            for img in page.get_images(full=True):
                xref = img[0]
                rects = page.get_image_rects(xref)
                for r in rects:
                    ratio = (r.width * r.height) / page_area
                    if ratio > max_img_ratio:
                        max_img_ratio = ratio
        except Exception:
            pass

        out.append({"text": text, "max_img_ratio": float(max_img_ratio)})

    doc.close()
    return out


CKPT = OUT_DIR / "ckpt_pymupdf.pkl"

if CKPT.exists():
    pages_data = pickle.loads(CKPT.read_bytes())
    print(f"Загружен чекпоинт ({len(pages_data)} docs)")
else:
    pages_data = {}
    t0 = time.time()
    for i, p in enumerate(pdf_paths):
        path_str = str(p)
        pages = extract_pages_pymupdf(path_str)
        pages_data[path_str] = {"total": len(pages), "pages": pages}
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(pdf_paths)}  ({time.time()-t0:.0f}s)")
    CKPT.write_bytes(pickle.dumps(pages_data))
    print(f"PyMuPDF: {len(pages_data)} docs за {time.time()-t0:.0f}s")

  50/203  (81s)
  100/203  (164s)
  150/203  (202s)
  200/203  (224s)
PyMuPDF: 203 docs за 227s


In [ ]:
SCAN_MARKERS_RE = re.compile(
    r"scanned\s+(by|with|using)|"
    r"camscanner|tap[\s\-]?scanner|adobe\s+scan|"
    r"сканировано|отсканировано",
    re.IGNORECASE,
)

LARGE_IMG_RATIO = 0.50
SCAN_MARKER_MAX_LEN = 200


def page_is_gap(page_info: dict) -> tuple[bool, str]:
    text = (page_info.get("text") or "").strip()
    img_ratio = page_info.get("max_img_ratio", 0.0)

    if len(text) < MIN_CHARS:
        return True, "empty_text"

    if len(text) <= SCAN_MARKER_MAX_LEN and SCAN_MARKERS_RE.search(text):
        return True, "scan_marker"

    if img_ratio >= LARGE_IMG_RATIO and len(text) < 400:
        return True, "image_heavy"

    return False, ""


stats = []
for path_str, info in pages_data.items():
    pages = info["pages"]
    total = info["total"]

    gap_indices = []
    reasons = []
    non_empty = 0
    for i, pg in enumerate(pages):
        is_gap, reason = page_is_gap(pg)
        if is_gap:
            gap_indices.append(i)
            reasons.append(reason)
        else:
            non_empty += 1

    stats.append({
        "path": path_str,
        "total_pages": total,
        "non_empty_pages": non_empty,
        "gap_count": len(gap_indices),
        "gap_indices": gap_indices,
        "gap_reasons": reasons,
    })

df_stats = pd.DataFrame(stats)
df = df.merge(df_stats, on="path")

df["text_pymupdf"] = df["path"].apply(
    lambda p: "\n".join(pg["text"] for pg in pages_data[p]["pages"])
)
df[["path", "filename", "year", "text_pymupdf"]].to_pickle(OUT_DIR / "df_pymupdf_only.pkl")

docs_with_gaps = df[df["gap_count"] > 0]
total_gap_pages = int(df["gap_count"].sum())
print(f"Документов с пробелами: {len(docs_with_gaps)} / {len(df)}")
print(f"Всего страниц-пробелов: {total_gap_pages}")

from collections import Counter
all_reasons = Counter()
for rs in df_stats["gap_reasons"]:
    all_reasons.update(rs)
print(f"Причины: {dict(all_reasons)}")
print(f"Saved {OUT_DIR / 'df_pymupdf_only.pkl'}")

Документов с пробелами: 157 / 203
Всего страниц-пробелов: 868
Причины: {'empty_text': 833, 'image_heavy': 35}
Saved artifacts\df_pymupdf_only.pkl


In [ ]:
TESS_CONFIG = "--oem 1 --psm 3"


def page_to_image(path: str, page_num: int, dpi: int = OCR_DPI) -> Image.Image:
    doc = fitz.open(path)
    pix = doc[page_num].get_pixmap(dpi=dpi, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    return img


def ocr_page(path: str, page_num: int) -> str:
    img = page_to_image(path, page_num)
    try:
        return pytesseract.image_to_string(img, lang=OCR_LANG, config=TESS_CONFIG)
    except pytesseract.TesseractError as e:
        print(f"[OCR ERR] {path} p.{page_num}: {e}")
        return ""


if len(docs_with_gaps) > 0:
    _row = docs_with_gaps.iloc[0]
    _pg = _row["gap_indices"][0]
    _sample = ocr_page(_row["path"], _pg)
    print(f"--- Sample OCR: {Path(_row['path']).name} стр. {_pg+1} ---")
    print(_sample[:500])
    print("---")

--- Sample OCR: Tursynbek D.Кітаптарды онлайн жалға алуды ұйымдастыратын веб-сайт құру.2019.pdf стр. 1 ---
ҚАЗАҚСТАН РЕСПУБЛИКАСЫНЫҢ БІЛІМ ЖӘНЕ ҒЫЛЫМ МИНИСТРЛІГІ
СӘТБАЕВ УНИВЕРСИТЕТІ

Қ.И. Сәтбаев атындағы Қазақ ұлттық техникалық зерттеу университеті
Мамандығы 5В070400 — ВТИПО
Студент Тұрсынбек Дәулет

Тақырыбы: «Кітаптарды онлайн жалға алуды ұйымдастыратын веб-сайт
құру»

ҒЫЛЫМИ ЖЕТЕКШІНІҢ
СЫН-ПІКІРІ

Диплом жобасын жасаушы Тұрсынбек Дәулеттің кітаптарды онлайн
жалға алуды ұйымдастыратын веб-сайт құру, брондау жүйесін енгізу,
администратор бетін және пайдаланушының жалға алған кітаптар тізімі бетін

---


In [7]:
CKPT_OCR = OUT_DIR / "ckpt_ocr_tess.pkl"

if CKPT_OCR.exists():
    ocr_results = pickle.loads(CKPT_OCR.read_bytes())
    print(f"Загружен OCR чекпоинт ({len(ocr_results)} docs)")
else:
    ocr_results = {}

t0 = time.time()
done_pages = 0
todo_docs = [r for _, r in docs_with_gaps.iterrows() if r["path"] not in ocr_results]
print(f"К обработке: {len(todo_docs)} документов")

for d_idx, row in enumerate(todo_docs):
    path_str = row["path"]
    gap_indices = row["gap_indices"]
    total = row["total_pages"]
    ocr_pages = {}

    for pg_idx in gap_indices:
        if pg_idx >= total:
            continue
        try:
            ocr_pages[pg_idx] = ocr_page(path_str, pg_idx)
        except Exception as e:
            print(f"[ERR] {Path(path_str).name} p.{pg_idx}: {e}")
            ocr_pages[pg_idx] = ""
        done_pages += 1

    ocr_results[path_str] = ocr_pages

    if (d_idx + 1) % 5 == 0 or d_idx + 1 == len(todo_docs):
        elapsed = time.time() - t0
        rate = done_pages / max(elapsed, 1e-3)
        remain_pages = sum(
            len(r["gap_indices"]) for r in todo_docs[d_idx + 1:]
        )
        eta = remain_pages / max(rate, 1e-3)
        print(
            f"  doc {d_idx+1}/{len(todo_docs)}  "
            f"pages={done_pages}  rate={rate:.2f} p/s  ETA {eta/60:.1f}min"
        )
        CKPT_OCR.write_bytes(pickle.dumps(ocr_results))

CKPT_OCR.write_bytes(pickle.dumps(ocr_results))
print(
    f"OCR готово: {len(ocr_results)} docs, "
    f"{sum(len(v) for v in ocr_results.values())} страниц "
    f"за {(time.time()-t0)/60:.1f} мин"
)

К обработке: 157 документов
  doc 5/157  pages=29  rate=0.78 p/s  ETA 17.9min
  doc 10/157  pages=45  rate=0.85 p/s  ETA 16.2min
  doc 15/157  pages=70  rate=0.73 p/s  ETA 18.3min
  doc 20/157  pages=96  rate=0.73 p/s  ETA 17.6min
  doc 25/157  pages=120  rate=0.69 p/s  ETA 18.0min
  doc 30/157  pages=145  rate=0.70 p/s  ETA 17.2min
  doc 35/157  pages=169  rate=0.73 p/s  ETA 15.9min
  doc 40/157  pages=197  rate=0.75 p/s  ETA 15.0min
  doc 45/157  pages=222  rate=0.73 p/s  ETA 14.7min
  doc 50/157  pages=248  rate=0.73 p/s  ETA 14.2min
  doc 55/157  pages=273  rate=0.73 p/s  ETA 13.6min
  doc 60/157  pages=349  rate=0.68 p/s  ETA 12.7min
  doc 65/157  pages=377  rate=0.67 p/s  ETA 12.3min
  doc 70/157  pages=401  rate=0.66 p/s  ETA 11.8min
  doc 75/157  pages=420  rate=0.67 p/s  ETA 11.2min
  doc 80/157  pages=434  rate=0.66 p/s  ETA 10.9min
  doc 85/157  pages=454  rate=0.65 p/s  ETA 10.6min
  doc 90/157  pages=474  rate=0.65 p/s  ETA 10.1min
  doc 95/157  pages=490  rate=0.65 p/s  E

In [8]:
def merge_text(path_str: str) -> str:
    info = pages_data[path_str]
    pages = info["pages"]
    ocr_pages = ocr_results.get(path_str, {})

    merged = []
    for i, pg in enumerate(pages):
        text = (pg.get("text") or "").strip()
        is_gap, _ = page_is_gap(pg)
        if is_gap and i in ocr_pages and ocr_pages[i].strip():
            merged.append(ocr_pages[i])
        else:
            merged.append(text)
    return "\n".join(merged)


df["text"]     = df["path"].apply(merge_text)
df["text_len"] = df["text"].str.len()

print(f"Пустых документов: {(df['text_len'] == 0).sum()}")
print(f"Средняя длина текста: {df['text_len'].mean():.0f} символов")
print(f"Медиана:              {df['text_len'].median():.0f} символов")

Пустых документов: 0
Средняя длина текста: 62647 символов
Медиана:              62433 символов


In [9]:
out_path = OUT_DIR / "df_extracted.pkl"
df.to_pickle(out_path)
print(f"Saved {out_path} ({len(df)} docs)")
print(f"Columns: {list(df.columns)}")

Saved artifacts\df_extracted.pkl (203 docs)
Columns: ['path', 'filename', 'year', 'total_pages', 'non_empty_pages', 'gap_count', 'gap_indices', 'gap_reasons', 'text_pymupdf', 'text', 'text_len']
